# White Castle rules assistant

RAG agent that answers questions about the rules of the board game White Castle:
- Knowledge base: the Spanish rulebook, indexed in ChromaDB with Gemini embeddings
- Agent: LangGraph graph (retrieve → generate) with Gemini and conversation memory

In [ ]:
# Imports
import base64
import os
import re
import statistics
from pathlib import Path

from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_core.messages import HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
from pypdf import PdfReader

In [ ]:
# Paths
PROJECT_DIR = Path.cwd()
ENV_PATH = PROJECT_DIR / ".env"
PDF_PATH = PROJECT_DIR / "data" / "white_castle.pdf"
TRANSCRIPT_DIR = PROJECT_DIR / "data" / "transcript"
CHROMA_DIR = PROJECT_DIR / "chroma_db"

# Models
CHAT_MODEL = "gemini-3.5-flash-lite"
EMBEDDING_MODEL = "gemini-embedding-2"

## Setup

Loads the Gemini API key from `.env` (variable `GOOGLE_API_KEY`). The key is never printed.

In [3]:
def load_api_key():
    """Gemini API key: .env → environment variable GOOGLE_API_KEY."""
    load_dotenv(ENV_PATH)
    if not os.getenv("GOOGLE_API_KEY"):
        raise ValueError(f"GOOGLE_API_KEY is empty or missing in {ENV_PATH}")


load_api_key()

## Smoke test

Checks that both models answer before building anything on top of them.

In [4]:
def check_chat():
    """Chat model check: short prompt → printed reply."""
    llm = ChatGoogleGenerativeAI(model=CHAT_MODEL)
    reply = llm.invoke("Reply in one short sentence: are you working?")
    print(f"{CHAT_MODEL}: {reply.text}")


def check_embeddings():
    """Embedding model check: short text → printed vector size."""
    embeddings = GoogleGenerativeAIEmbeddings(model=EMBEDDING_MODEL)
    vector = embeddings.embed_query("White Castle")
    print(f"{EMBEDDING_MODEL}: vector of size {len(vector)}")


check_chat()
check_embeddings()

gemini-3.5-flash-lite: Yes, I am currently active and ready to assist you.
gemini-embedding-2: vector of size 3072


## 1. Transcription

The rulebook PDF is scanned: every page is a single image with no text layer. Gemini reads each page image and transcribes it:
- One request per page → one Markdown file per page in `data/transcript/`
- Pages already transcribed are skipped, so the quota is only spent once
- Icons and illustrated examples that carry rules are described briefly between square brackets

In [ ]:
# Constants
TRANSCRIPTION_PROMPT = """You are transcribing one page of the Spanish rulebook of the board game White Castle.
- Transcribe all the text literally, in Spanish. Do not translate, summarize or correct it.
- Mark section titles with Markdown headings: # for main titles, ## for subsections.
- When an icon, diagram or illustrated example carries rules information, describe it briefly in Spanish between square brackets, for example: [Icono: 2 monedas].
- Ignore purely decorative illustrations, page numbers and repeated headers or footers.
- Return only the transcription, with no comments before or after it."""

In [ ]:
def transcribe_manual():
    """Rulebook PDF → one Markdown file per page in TRANSCRIPT_DIR (pages already done are skipped)."""
    TRANSCRIPT_DIR.mkdir(parents=True, exist_ok=True)
    llm = ChatGoogleGenerativeAI(model=CHAT_MODEL)
    pages = PdfReader(PDF_PATH).pages
    for number, page in enumerate(pages, start=1):
        transcribe_page(llm, page, number)


def transcribe_page(llm, page, number):
    """One page: scanned image → Gemini transcription → page_NN.md."""
    output_path = TRANSCRIPT_DIR / f"page_{number:02d}.md"
    if output_path.exists():
        print(f"Page {number:02d}: already transcribed, skipped")
        return
    message = build_page_message(page)
    reply = llm.invoke([message])
    output_path.write_text(reply.text, encoding="utf-8")
    print(f"Page {number:02d}: transcribed ({len(reply.text)} characters)")


def build_page_message(page):
    """Message for Gemini: transcription prompt + the page JPEG image as base64."""
    image_b64 = base64.b64encode(page.images[0].data).decode()
    return HumanMessage(content=[
        {"type": "text", "text": TRANSCRIPTION_PROMPT},
        {"type": "image_url", "image_url": f"data:image/jpeg;base64,{image_b64}"},
    ])


transcribe_manual()

## 2. Cleaning and chunking

The transcription is turned into chunks, the pieces the agent will search:
- Cleaning: removes transcription noise (formula marks, arrow bullets, decorative images)
- Sections first: splits by Markdown headings so a rule is not cut in two. A page that starts mid-section keeps the section of the previous page
- Then size: long sections are split into chunks with some overlap
- Each chunk keeps its page and section, and starts with its section title so the embedding knows the context

In [ ]:
# Constants
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150

HEADERS = [("#", "h1"), ("##", "h2"), ("###", "h3")]

# Cleaning rules: (regex pattern, replacement)
CLEANING_RULES = [
    (r"\$([^$\n]+)\$",                        r"\1"),  # formula marks: $10+10$ → 10+10
    (r"\[Icono: Flecha roja\]\s*",            "- "),   # arrow bullets → list items
    (r"\[Icono: Signo de exclamación\]\s*",   ""),     # warning sign before "Atención"
    (r"\*?\[Ilustración decorativa:[^\]]*\]\*?", ""),  # decorative images
    (r"\n{3,}",                               "\n\n"), # extra blank lines
]

In [ ]:
def build_chunks():
    """Transcript pages → cleaned text → sections → chunks with page and section metadata."""
    pages = load_pages()
    sections = split_into_sections(pages)
    chunks = split_long_sections(sections)
    print_chunk_summary(chunks)
    return chunks


def load_pages():
    """Transcript files → list of (page number, cleaned text)."""
    pages = []
    for path in sorted(TRANSCRIPT_DIR.glob("page_*.md")):
        number = int(path.stem.split("_")[1])
        pages.append((number, clean_text(path.read_text(encoding="utf-8"))))
    return pages


def clean_text(text):
    """Raw transcription → text without the noise listed in CLEANING_RULES."""
    for pattern, replacement in CLEANING_RULES:
        text = re.sub(pattern, replacement, text)
    return text.strip()


def split_into_sections(pages):
    """
    Pages → one document per section:
        - Metadata: headings (h1, h2, h3) and page number
        - Text at the top of a page with no heading keeps the section of the previous page
    """
    splitter = MarkdownHeaderTextSplitter(headers_to_split_on=HEADERS)
    sections = []
    last_headers = {}
    for number, text in pages:
        for section in splitter.split_text(text):
            if section.metadata:
                last_headers = dict(section.metadata)
            else:
                section.metadata = dict(last_headers)
            section.metadata["section"] = " > ".join(section.metadata.values())
            section.metadata["page"] = number
            sections.append(section)
    return sections


def split_long_sections(sections):
    """Sections → chunks of at most CHUNK_SIZE characters, each starting with its section title."""
    splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
    chunks = splitter.split_documents(sections)
    for chunk in chunks:
        chunk.page_content = f"{chunk.metadata['section']}\n\n{chunk.page_content}"
    return chunks


def print_chunk_summary(chunks):
    """Chunk count, length statistics and one example chunk."""
    lengths = [len(chunk.page_content) for chunk in chunks]
    print(f"Chunks: {len(chunks)}")
    print(f"Length (characters): min {min(lengths)}, median {statistics.median(lengths):.0f}, max {max(lengths)}")
    print(f"\nExample (chunk 10, page {chunks[10].metadata['page']}):\n{chunks[10].page_content}")


chunks = build_chunks()

## 3. Vector store

Each chunk is turned into a vector with Gemini embeddings and stored in a local ChromaDB collection (`chroma_db/`):
- If the collection already exists, it is loaded from disk and no embeddings are spent
- After changing the cleaning or chunking settings, set `REBUILD_VECTOR_STORE = True` once to rebuild it

In [ ]:
# Constants
COLLECTION_NAME = "white_castle_rules"
REBUILD_VECTOR_STORE = False

In [ ]:
def load_vector_store(chunks):
    """Chunks → ChromaDB collection in CHROMA_DIR (loaded from disk if it is already built)."""
    vector_store = Chroma(
        collection_name=COLLECTION_NAME,
        embedding_function=GoogleGenerativeAIEmbeddings(model=EMBEDDING_MODEL),
        persist_directory=str(CHROMA_DIR),
    )
    stored = len(vector_store.get()["ids"])
    if REBUILD_VECTOR_STORE or stored == 0:
        index_chunks(vector_store, chunks)
    else:
        print(f"Vector store loaded from disk: {stored} chunks")
    return vector_store


def index_chunks(vector_store, chunks):
    """Empty the collection, then embed and store every chunk with a stable id."""
    vector_store.reset_collection()
    ids = [f"chunk_{number:03d}" for number in range(len(chunks))]
    vector_store.add_documents(chunks, ids=ids)
    print(f"Vector store built: {len(chunks)} chunks indexed")


vector_store = load_vector_store(chunks)

## 4. Retrieval check

Before connecting the agent, test questions with a known answer in the rulebook check that the right chunks come back. For each question: the top chunks with their distance (lower = more similar), page and section.

In [ ]:
# Constants
RETRIEVAL_K = 3

# Test questions: (question, where the answer is in the rulebook)
TEST_QUESTIONS = [
    ("¿Se pueden apilar dados en una partida a 2 jugadores?",              "p5, Apilar dados"),
    ("¿Qué pasa si coloco un dado de valor inferior al de la casilla?",     "p5, Uso de los dados"),
    ("¿Cuántos cortesanos caben en la puerta del castillo?",                "p10, Cortesanos"),
    ("¿Cuál es el máximo de recursos de cada tipo que puedo tener?",        "p5, Obtener y pagar recursos"),
    ("¿Cuánto cuesta subir un cortesano dos pisos?",                        "p10, Cortesanos"),
    ("¿Cómo puntúan los guerreros al final de la partida?",                 "p14, Final de partida"),
    ("En el modo solitario, ¿el clan rival puntúa los recursos sobrantes?", "p18, Puntuación final"),
]

In [ ]:
def check_retrieval():
    """Every test question → top RETRIEVAL_K chunks printed next to the expected location."""
    for question, expected in TEST_QUESTIONS:
        print(f"\n{question}\n  Expected: {expected}")
        for chunk, distance in vector_store.similarity_search_with_score(question, k=RETRIEVAL_K):
            print(f"  [{distance:.3f}] p{chunk.metadata['page']:02d} | {chunk.metadata['section'][:70]}")


check_retrieval()